# Hull Tactical - Market Prediction Japanese Tutorial (日本語チュートリアル)

## Hull Tactical - Market Prediction 概要

### コンペティション概要
- **主催者**: Hull Tactical（クオンツ投資戦略を展開する企業）
- **目的**: 金融市場（特に S&P500 指数）の予測可能性を検証するための挑戦
- **期間**:  
  - 開始: 2025年9月15日  
  - 提出締切: 2025年12月15日  
  - コンペ終了: 2026年6月16日（最終評価を含む）
- **賞金総額**: 100,000ドル（1位 50,000ドル）
- **特徴**: 効率的市場仮説 (Efficient Market Hypothesis) に挑む設計で、実運用性を考慮した評価が行われる。

---

### 課題設定・目的
- **対象**: 米国株式インデックス（S&P 500）の超過リターン
- **タスク**: 市場リターンの予測
- **評価指標**: 修正 Sharpe 比 (modified Sharpe ratio)  
  - リスクを抑えつつ安定的にリターンを獲得できるかを重視
- **制約**: ボラティリティ（変動性）を過剰に高めるだけの戦略は評価されない
- **意図**: 予測精度だけでなく、投資戦略としての持続性・実効性が求められる

---

### モチベーションとチャレンジ点
1. **低シグナル対雑音**  
   市場データはノイズが多く、有意な予測シグナルを見つけることが難しい。  

2. **リスクとリターンのトレードオフ**  
   高リターンを追うだけでは不十分で、ボラティリティ制御が不可欠。  

3. **過剰適合のリスク**  
   特徴量やモデルを複雑にすると、学習データに合っても将来データで性能が落ちる。  

4. **モデルの解釈性・頑健性**  
   実運用を意識すると、極端な出力や不安定な戦略は不利になる。  

5. **評価フェーズの特異性**  
   単なる予測精度だけでなく、**実際の市場リターンとの比較**が評価に組み込まれる。


### 準備

In [25]:
# ライブラリーインポート
#　ライブラリー(いつもの)
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

#　ライブラリー(追加)
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ディスプレイオプション
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set professional plotting style
plt.style.use('ggplot')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['font.family'] = 'Arial'
custom_palette = ["#3498db", "#e74c3c", "#2ecc71", "#f39c12", "#9b59b6"]
sns.set_palette(custom_palette)

In [26]:
# データ読み込み
BASE_DIR = "./data"
# BASE_DIR = "/kaggle/input/Hull Tactical - Market Prediction"
train_df = pl.read_csv(BASE_DIR + "/train.csv")
test_df = pl.read_csv(BASE_DIR + "/test.csv")

datasets = {
    "train data" : train_df,
    "test data"  : test_df
}

## データ初期調査

## データの概要 (Data Overview)

### 提供データの構成 

主なファイルは以下のとおりです：

- **train.csv** : 学習用データセット（過去の市場データとターゲットラベル）
- **test.csv** : 評価用データセット（将来期間の市場データ、ターゲットは非公開）

---

### カラム詳細(train.csv)
- *date_id* : 1 取引日の識別子。
- *M\** : 市場の動向/技術的特徴。
- *E\** : マクロ経済の特徴。
- *I\** : 金利機能。
- *P\** : 価格/評価機能。
- *V\** : ボラティリティ機能。
- *S\** : 感情機能。
- *MOM\** : モメンタム機能。
- *D\** : ダミー/バイナリ機能。
- *forward_returns* : S&P 500を購入し、翌日に売却した場合のリターン。列車セットのみ。
- *risk_free_rate* : フェデラルファンド金利。トレーニングセットのみ。
- *market_forward_excess_returns* : 期待値に対するフォワードリターン。5年間のローリング平均フォワードリターンを差し引き、その結果を基準値4の中央絶対偏差（MAD）を用いてウィンザライズすることで算出。トレーニングセットのみ。

### カラム詳細(test.csv)
- *date_id, [feature_name]* : 特徴列は と同じですtrain.csv。
- *is_scored* : この行が評価指標の計算に含まれるかどうか。モデルのトレーニングフェーズでは、最初の180行のみ対象となります。テストセットのみ。
- *lagged_forward_returns* : S&P 500 を購入し、1 日後に売却することで得られるリターン (1 日の遅延あり)。
- *lagged_risk_free_rate* : 1 日遅れのフェデラルファンド金利。
- *lagged_market_forward_excess_returns* : 期待値に対するフォワードリターン。5年間の平均フォワードリターンを差し引き、基準値4の中央絶対偏差（MAD）を用いて1日遅れでウィンザー化して算出。

In [27]:
print("<Info>")
for name, df in datasets.items():
    print(f"{name}:")
    display(df.info())

<Info>
train data:


AttributeError: 'DataFrame' object has no attribute 'info'

In [28]:
print("<Shape>")
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

<Shape>
train data: (8990, 98)
test data: (10, 99)


In [30]:
print("<Nan>")
for name, df in datasets.items():
    print(f"{name}:")
    display(df.isna().sum())

<Nan>
train data:


AttributeError: 'DataFrame' object has no attribute 'isna'

df.info()ではNon-nullは存在しないと出力されたが、データにはNanという文字列が含まれているため欠損値が存在するということとなる

In [31]:
print("Raw Data")
for name, df in datasets.items():
    print(f"<{name}>")
    display(df.sample(5))

Raw Data
<train data>


date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,I7,…,P13,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64
6493,0,0,0,0,0,0,0,0,0,"""1.47757091148905""","""0.99239417989418""","""0.0188492063492063""","""0.00562169312169312""","""0.00562169312169312""","""0.0188492063492063""","""0.313822751322751""","""-1.59785960825912""","""-1.69477871903962""","""-1.6956653911624""","""0.585859009262659""","""0.645402923528086""","""0.53927918678578""","""0.368333885499341""","""0.00297619047619048""","""0.865740740740741""","""0.193121693121693""",null,"""-1.40860655074663""","""0.00297619047619048""","""0.841931216931217""","""0.117157253044052""","""0.0975529100529101""","""0.583002645502645""","""-1.14793922462516""","""0.10978835978836""","""0.830687830687831""",…,"""0.0992063492063492""","""1.61050428860359""","""0.465939153439153""","""0.105489417989418""","""-1.03147650936975""","""-0.414377326884686""","""1.36988961363865""","""1.75459167452743""","""0.227513227513228""","""-0.762043735003009""","""0.080026455026455""","""0.0231481481481481""","""2.14458228226266""","""-0.911609455298974""","""-1.1685265213924""","""0.0648148148148148""","""-1.50079757458213""","""0.359126984126984""","""0.426256613756614""","""-1.48553390685046""","""0.990079365079365""","""0.908068783068783""","""-0.107350081879275""","""0.793650793650794""","""0.0806878306878307""","""-0.688775112745672""","""0.794973544973545""","""0.854497354497355""","""0.743386243386243""","""1.00631907964292""","""0.206349206349206""","""-0.0189530683215655""","""0.833664021164021""","""0.0716262025485329""",0.009027,1.9841e-7,0.008719
5322,0,0,0,1,0,0,0,0,0,"""2.4212955836212""","""0.0651455026455026""","""0.574404761904762""","""0.315145502645503""","""0.0320767195767196""","""0.00297619047619048""","""0.957010582010582""","""1.12029825343661""","""1.31213905294153""","""1.12841522336657""","""-0.236729216779201""","""0.0959912304854304""","""0.892119779468105""","""-0.365194472287875""","""0.0158730158730159""","""0.666997354497355""","""0.00363756613756614""",null,"""-0.19771163126355""","""0.374007936507937""","""0.14484126984127""","""-0.598698570187866""","""0.860449735449735""","""0.486111111111111""","""-1.09702296563554""","""0.474206349206349""","""0.544312169312169""",…,"""0.712301587301587""","""-0.730324833884293""","""0.816468253968254""","""0.522156084656085""","""0.736326083485026""","""-0.332290554331257""","""-0.054480339931293""","""1.34067974931093""","""0.888227513227513""","""-1.18497527312874""","""0.429232804232804""","""0.66468253968254""","""0.380496214666911""","""1.09949250544688""",null,"""0.92526455026455""","""-0.0422135395791018""","""0.713293650793651""","""0.213624338624339""","""1.43173309306567""","""0.0476190476190476""","""0.130291005291005""",null,"""0.0191798941798942""","""0.685846560846561""","""-0.358894219728652""","""0.183862433862434""","""0.21031746031746""","""0.233465608465608""","""1.99843709907952""","""0.226851851851852""","""-0.488065418683951""","""0.685846560846561""","""-0.428370913918045""",0.005963,0.000004,0.005651
4278,0,0,1,1,0,0,0,0,0,"""1.36728290623979""","""0.815145502645503""","""0.0281084656084656""","""0.0446428571428571""","""0.00363756613756614""","""0.00363756613756614""","""0.997685185185185""","""-0.727403226059443""","""-0.307661363367511""","""-0.638014335126843""","""-1.06289276296708""","""-0.146465404632062""","""1.70739781989505""","""-0.292647502150073""","""0.00165343915343915""","""0.36276455026455""","""0.011

<test data>


date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,I7,…,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,is_scored,lagged_forward_returns,lagged_risk_free_rate,lagged_market_forward_excess_returns
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64
8984,0,0,0,0,0,0,1,0,1,1.567818,0.184854,0.019511,0.019511,0.006283,0.006283,0.956349,-0.083449,-0.572813,0.224366,-0.714549,1.243713,1.542543,1.693604,0.031746,0.332011,0.035053,0.029586,-0.301855,0.914352,0.260913,1.011571,0.093915,0.476852,0.638667,0.729497,0.238757,…,-1.416282,0.611442,0.426257,0.046472,-0.262086,1.304142,1.785515,0.039683,0.230364,0.583995,0.445106,1.378927,-0.182025,0.01261,0.369048,0.011021,0.130291,0.409392,0.572656,0.489418,0.600529,-0.587258,0.46131,0.487434,-0.39548,0.80754,0.707672,0.839947,0.944593,0.715608,-0.692649,0.124669,-0.654045,true,0.008357,0.000159,0.007887
8989,0,0,0,0,0,0,0,0,0,1.55569,0.183201,0.017857,0.017857,0.00463,0.00463,0.912698,-0.084538,-0.571888,0.220814,-0.747066,1.312626,1.529875,1.765317,0.030093,0.330357,0.033399,0.037561,-0.267677,0.916005,0.305886,0.822594,0.080688,0.478175,0.603072,0.705688,0.263889,…,-1.469985,0.553571,0.244378,0.306917,-0.177546,1.885541,1.796492,0.047619,0.410794,0.351852,0.070767,0.257443,0.011109,-0.64248,0.170635,-0.105022,0.354167,0.409392,0.629295,0.771825,0.318783,-0.66805,0.12963,0.37004,-0.451308,0.66336,0.066799,0.78373,1.068037,0.87963,-0.764806,0.079034,-0.705662,true,0.00831,0.000156,0.007843
8987,0,0,1,0,0,0,0,0,0,1.56052,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,-0.800465,1.247273,1.534742,1.695469,0.030754,0.331019,0.034061,0.038118,-0.301918,0.915344,0.273148,0.842295,0.074074,0.478836,0.611319,0.724868,0.223545,…,-1.420007,0.849868,0.375661,0.429471,-0.233374,-0.289766,1.792816,0.041005,0.371362,0.793651,0.689815,0.885178,-0.316882,-0.422374,0.631614,-0.02977,0.221892,0.409392,0.583556,0.477513,0.599206,-0.638658,0.394841,0.433532,-0.425462,0.734127,0.481481,0.787698,0.834898,0.823413,-0.723949,0.133929,-0.670946,true,0.002312,0.000156,0.001845
8980,0,0,0,0,1,0,0,1,0,1.577651,0.186177,0.001323,0.001323,0.001323,0.001323,0.955026,-0.583419,-0.704264,0.298365,-0.691361,1.259065,1.556516,1.71258,0.033069,0.333333,0.036376,-0.046483,-0.312326,0.913029,0.306217,1.025756,0.081349,0.478175,0.675627,0.699735,0.256283,…,-1.427834,0.352513,0.926257,0.431383,-0.476976,0.500245,1.784173,0.029762,0.294719,0.51455,0.446429,0.466551,0.085717,-0.230132,0.272487,-0.106894,0.199735,0.409392,0.532717,0.744048,0.440476,-0.654839,0.699735,0.699074,-0.5024,0.882937,0.892196,0.828042,0.999172,0.759921,-0.803127,0.170966,-0.751909,true,0.003541,0.000161,0.003068
8983,0,0,0,0,1,0,0,0,1,1.570266,0.185185,0.019841,0.019841,0.006614,0.006614,0.956019,-0.083403,-0.57318,0.225094,-0.541646,1.167982,1.549272,1.611215,0.032077,0.332341,0.035384,0.033998,-0.312383,0.914021,0.267196,1.091376,0.085979,0.477183,0.648582,0.728175,0.230159,…,-1.399411,0.724868,0.21627,-0.230557,-0.318347,0.396917,1.769678,0.03373,0.20235,0.324074,0.212963,0.052878,-0.049023,0.120828,0.219577,0.137942,0.167328,0.409392,0.579726,0.449735,0.665344,-0.546298,0.590608,0.558862,-0.275099,0.826058,0.445767,0.835979,1.040988,0.594577,-0.561643,0.161706,-0.575997,true,0.00542,0.00016,0.004949


In [32]:
print("<Describe>")
for name, df in datasets.items():
    print(f"{name}")
    display(df.describe())

<Describe>
train data


statistic,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,…,P13,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64
"""count""",8990.0,8990.0,8990.0,8990.0,8990.0,8990.0,8990.0,8990.0,8990.0,8990.0,"""7206""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7374""","""7984""","""7984""","""7984""","""7984""","""2021""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""","""7984""",…,"""7984""","""7984""","""7984""","""7984""","""7416""","""7352""","""7374""","""7984""","""7984""","""7984""","""7984""","""7984""","""5453""","""7984""","""3257""","""7984""","""7479""","""7984""","""7984""","""5981""","""7984""","""7984""","""2941""","""7984""","""7984""","""7479""","""7984""","""7984""","""7984""","""7478""","""7984""","""7479""","""7984""","""4451""",8990.0,8990.0,8990.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""1784""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1616""","""1006""","""1006""","""1006""","""1006""","""6969""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""","""1006""",…,"""1006""","""1006""","""1006""","""1006""","""1574""","""1638""","""1616""","""1006""","""1006""","""1006""","""1006""","""1006""","""3537""","""1006""","""5733""","""1006""","""1511""","""1006""","""1006""","""3009""","""1006""","""1006""","""6049""","""1006""","""1006""","""1511""","""1006""","""1006""","""1006""","""1512""","""1006""","""1511""","""1006""","""4539""",0.0,0.0,0.0
"""mean""",4494.5,0.031591,0.031591,0.047831,0.575195,0.190656,-0.238042,0.045717,0.142825,0.143159,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.000469,0.000107,0.000051
"""std""",2595.333794,0.174917,0.174917,0.21342,0.494341,0.39284,0.425909,0.208883,0.349914,0.350254,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.010551,0.000088,0.010568
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,0.0,"""0.325149307781904""","""0.000661375661375661""","""0.000661375661375661""","""0.000661375661375661""","""0.000661375661375661""","""0.000661375661375661""","""0.000661375661375661""","""-0.000106995292857562""","""-0.000216543653035166""","""-0.0370537162477823""","""-0.00103514761966728""","""-0.000286476485653406""","""-0.0148789892924795""","""-0.000109469415770866""","""0.000661375661375661""","""0.000661375661375661""","""0.000661375661375661""","""-0.00191866426052122""","""-0.000332320536729127""","""0.000661375661375661""","""0.00264550264550265""","""-0.000158919870282691""","""0.000661375661375661""","""0.000661375661375661""","""-0.00107541867648806""","""0.000661375661375661""",…,"""0.00231481481481481""","""-0.000380178000909207""","""0.0439814814814815""","""0.0661375661375661""","""-0.0000605148090623544""","""-0.000612006826255119""","""-0.0000466649000800819""","""-0.00013672794415026""","""0.000661375661375661""","""-0.000096336366423328""","""0.00066137

test data


statistic,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,E19,E2,E20,E3,E4,E5,E6,E7,E8,E9,I1,I2,I3,I4,I5,I6,…,P2,P3,P4,P5,P6,P7,P8,P9,S1,S10,S11,S12,S2,S3,S4,S5,S6,S7,S8,S9,V1,V10,V11,V12,V13,V2,V3,V4,V5,V6,V7,V8,V9,is_scored,lagged_forward_returns,lagged_risk_free_rate,lagged_market_forward_excess_returns
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,…,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",8984.5,0.0,0.0,0.1,0.0,0.4,0.0,0.1,0.2,0.3,1.566627,0.184689,0.013492,0.013492,0.004233,0.004233,0.933862,-0.183636,-0.598795,0.238234,-0.584514,1.225412,1.542907,1.673009,0.031581,0.331845,0.034888,0.031407,-0.302658,0.914517,0.282804,0.957083,0.081614,0.477877,0.630171,0.718783,…,-1.402797,0.483796,0.417361,0.188995,-0.32594,0.444488,1.782248,0.038161,0.303177,0.450661,0.353704,0.653249,-0.070573,-0.205986,0.296627,0.045692,0.218122,0.409392,0.587802,0.661772,0.543981,-0.604714,0.505192,0.520569,-0.38614,0.787434,0.573743,0.819907,1.010369,0.747487,-0.679972,0.145238,-0.649404,1.0,0.001702,0.000158,0.001232
"""std""",3.02765,0.0,0.0,0.316228,0.0,0.516398,0.0,0.316228,0.421637,0.483046,0.007388,0.001001,0.008647,0.008647,0.002316,0.002316,0.023011,0.210614,0.055457,0.031531,0.205915,0.054477,0.009505,0.058473,0.001001,0.001001,0.001001,0.029955,0.013344,0.001001,0.017817,0.093216,0.005751,0.00086,0.028975,0.009461,…,0.037741,0.300459,0.238504,0.60086,0.106784,0.831943,0.013457,0.005998,0.07338,0.296879,0.266854,0.669503,0.259807,0.273719,0.199868,0.12097,0.059294,0.0,0.028391,0.194999,0.116091,0.056397,0.177948,0.102121,0.095649,0.072821,0.223494,0.024329,0.131349,0.111075,0.099564,0.038423,0.065991,null,0.005482,0.000002,0.005483
"""min""",8980.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.55569,0.183201,0.000661,0.000661,0.000661,0.000661,0.911376,-0.583419,-0.704264,0.220814,-0.800465,1.123361,1.529875,1.562722,0.030093,0.330357,0.033399,-0.046483,-0.312383,0.913029,0.260913,0.822594,0.073413,0.476521,0.597442,0.699735,…,-1.469985,0.046296,0.068783,-1.138198,-0.494248,-1.042718,1.754171,0.029762,0.20235,0.011905,0.026455,-0.201611,-0.446682,-0.64248,0.066138,-0.106894,0.130291,0.409392,0.532717,0.37037,0.318783,-0.66805,0.12963,0.37004,-0.5024,0.66336,0.066799,0.78373,0.785877,0.556217,-0.803127,0.079034,-0.751909,1.0,-0.00741,0.000155,-0.007882
"""25%""",8982.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.56052,0.183862,0.001323,0.001323,0.001323,0.001323,0.912037,-0.084538,-0.573546,0.222211,-0.732397,1.193468,1.534742,1.640054,0.030754,0.331019,0.034061,0.033581,-0.312345,0.91369,0.269841,0.858582,0.080688,0.477183,0.604658,0.718254,…,-1.427834,0.232143,0.244378,0.044888,-0.421365,-0.234829,1.770175,0.03373,0.249933,0.273148,0.134921,0.052878,-0.316882,-0.422374,0.137566,-0.02977,0.194444,0.409392,0.574661,0.477513,0.462302,-0.64204,0.394841,0.433532,-0.432282,0.734127,0.469577,0.787698,0.944593,0.665344,-0.723949,0.124669,-0.670946,null,-0.002896,0.000156,-0.003365
"""50%""",8985.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.567818,0.184854,0.018519,0.018519,0.005291,0.005291,0.955026,-0.083542,-0.572447,0.224366,-0.596939,1.24371

### コメント

今回はpandasの使用をやめてporalsを使用してEDAを行ってみました。